# MacroChefAI v5 — Revised End-to-End Notebook

This notebook is a streamlined, app-oriented revision of the recipe preprocessing and recommendation pipeline.

It is designed to support an app that:
- collects user inputs such as age, weight, height, sex, activity level, goal, pantry ingredients, must-include and must-exclude ingredients, health issues, maximum cooking time, meals per day, number of recommendations, and macro preferences
- returns recipe outputs such as Nutri-Score (`A` to `E`), macro labels, macro score, ingredient score, weight-loss probability, matched and missing ingredients, ingredients, and instruction steps
- sorts recommendations by macro score, ingredient score, weight-loss probability, or health score

The notebook also:
- keeps the **weight-loss score / label / probability** logic
- adds **Nutri-Score** features and label
- keeps **NRF-style health scoring**
- splits processed data into two parquet files to avoid large object-heavy files that can make VS Code unstable

In [4]:
"""Import libraries, define paths, and create output folders."""

import os
import ast
import json
import re
import warnings
import joblib
from fractions import Fraction
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.sparse import save_npz, load_npz
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore", category=SyntaxWarning)

EPS = 1e-9

if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

RAW_DATA_PATH = "raw_data/merged_usable_cal_per_g.csv"

PROCESSED_MODEL_PATH = "processed_data/recipes_model_ready.parquet"
PROCESSED_DISPLAY_PATH = "processed_data/recipes_display_ready.parquet"

WEIGHT_LOSS_MODEL_PATH = "models/weight_loss_pipeline.joblib"
TFIDF_PATH = "models/tfidf_vectorizer.joblib"
INGREDIENT_MATRIX_PATH = "models/ingredient_matrix.npz"

os.makedirs("processed_data", exist_ok=True)
os.makedirs("models", exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data path:", RAW_DATA_PATH)

Project root: /home/bekheet/code/magedbekheet/macrochefai
Raw data path: raw_data/merged_usable_cal_per_g.csv


## 1. Helper Parsing and Normalization Functions

In [5]:
"""Helper functions for parsing list-like columns, time strings, and ingredient text."""

def _safe_list(x):
    """Safely convert many list-like formats into Python lists.

    Handles:
    - existing Python lists
    - JSON-style list strings
    - Python-style list strings
    - R-style c("a","b")
    - simple comma-separated strings

    Avoids calling ast.literal_eval on arbitrary text such as:
    '1/2-1 lb corned beef', which can trigger SyntaxWarning.
    """
    if pd.isna(x):
        return []

    if isinstance(x, list):
        return x

    s = str(x).strip()
    if not s:
        return []

    if s.startswith("c(") and s.endswith(")"):
        inner = s[2:-1]
        parts = re.findall(r'"([^"]+)"|\'([^\']+)\'', inner)
        vals = [a or b for a, b in parts if (a or b)]
        return vals if vals else []

    if s.startswith("[") and s.endswith("]"):
        try:
            val = json.loads(s)
            if isinstance(val, list):
                return val
        except Exception:
            pass

        try:
            val = ast.literal_eval(s)
            if isinstance(val, list):
                return val
        except Exception:
            pass

    if "," in s:
        return [item.strip() for item in s.split(",") if item.strip()]

    return [s]


def parse_instructions(value):
    """Parse recipe instructions into a clean list of steps.

    Supports:
    - actual Python lists
    - JSON-style lists
    - Python-style list strings
    - R-style c("step1","step2")
    - plain text split by periods/newlines

    Avoids unsafe ast parsing on arbitrary non-list text.
    """
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]

    text = str(value).strip()
    if not text:
        return []

    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = json.loads(text)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass

        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass

    if text.startswith("c(") and text.endswith(")"):
        inner = text[2:-1]
        parts = re.findall(r'"([^"]+)"|\'([^\']+)\'', inner)
        vals = [(a or b).strip() for a, b in parts if (a or b)]
        if vals:
            return vals

    return [p.strip() for p in re.split(r"\.\s+|\n+", text) if p.strip()]


def parse_quantity_value(x):
    """Parse a quantity string into a float when possible.

    Examples:
    - '1 1/2' -> 1.5
    - '1/2-1' -> 0.75
    - '2' -> 2.0

    Parameters
    ----------
    x : Any
        Input quantity text.

    Returns
    -------
    float
        Parsed numeric quantity, or NaN when parsing fails.
    """
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()
    if not s or s in {"none", "nan", "na", "to taste", "as needed"}:
        return np.nan

    s = s.replace("–", "-").replace("—", "-")
    s = re.sub(r"\([^\)]*\)", "", s).strip()

    try:
        if " " in s:
            parts = s.split()
            if len(parts) == 2 and "/" in parts[1]:
                return float(parts[0]) + float(Fraction(parts[1]))
        if "-" in s:
            vals = [parse_quantity_value(part) for part in s.split("-") if part.strip()]
            vals = [v for v in vals if pd.notna(v)]
            if vals:
                return float(np.mean(vals))
        return float(Fraction(s))
    except Exception:
        try:
            return float(s)
        except Exception:
            return np.nan


def normalize_quantity_list(q_list):
    """Normalize a list-like quantity field into numeric values.

    Parameters
    ----------
    q_list : Any
        List-like quantity field.

    Returns
    -------
    list[float]
        Parsed numeric values with missing entries replaced by 0.0.
    """
    values = [parse_quantity_value(x) for x in _safe_list(q_list)]
    return [0.0 if pd.isna(v) else float(v) for v in values]


def parse_time_to_minutes(x):
    """Convert ISO-8601-like duration strings such as PT1H15M into minutes.

    Parameters
    ----------
    x : Any
        Raw time value.

    Returns
    -------
    float
        Minutes, or NaN when parsing fails.
    """
    if pd.isna(x):
        return np.nan

    s = str(x).strip()
    match = re.match(r"^PT(?:(\d+)H)?(?:(\d+)M)?$", s)
    if match:
        h = int(match.group(1) or 0)
        m = int(match.group(2) or 0)
        return h * 60 + m

    try:
        return float(s)
    except Exception:
        return np.nan

def build_display_ingredient_rows(row: pd.Series) -> list[dict]:
    """Build display-friendly ingredient rows with quantity, raw text, and normalized name."""
    raw_list = row.get("ingredients_raw_list", [])
    clean_list = row.get("ingredients_clean", [])
    qty_list = row.get("ingredient_quantities", [])

    raw_list = raw_list if isinstance(raw_list, list) else []
    clean_list = clean_list if isinstance(clean_list, list) else []
    qty_list = qty_list if isinstance(qty_list, list) else []

    n = max(len(raw_list), len(clean_list), len(qty_list))
    rows = []

    for i in range(n):
        rows.append({
            "quantity": qty_list[i] if i < len(qty_list) else None,
            "raw_ingredient": raw_list[i] if i < len(raw_list) else None,
            "normalized_ingredient": clean_list[i] if i < len(clean_list) else None,
        })

    return rows

In [7]:
"""Ingredient normalization dictionaries and helper functions."""

INGREDIENT_ALIASES = {
    "garbanzo bean": "chickpea",
    "garbanzo beans": "chickpea",
    "chickpeas": "chickpea",
    "black beans": "black bean",
    "kidney beans": "kidney bean",
    "white beans": "white bean",
    "cannellini beans": "cannellini bean",
    "spring onions": "green onion",
    "scallions": "green onion",
    "red onions": "red onion",
    "yellow onions": "yellow onion",
    "brown onions": "onion",
    "tomatoes": "tomato",
    "potatoes": "potato",
    "sweet potatoes": "sweet potato",
    "bell peppers": "bell pepper",
    "capsicum": "bell pepper",
    "courgette": "zucchini",
    "aubergine": "eggplant",
    "cilantro": "coriander",
    "confectioners sugar": "powdered sugar",
    "icing sugar": "powdered sugar",
    "caster sugar": "sugar",
    "extra virgin olive oil": "olive oil",
    "virgin olive oil": "olive oil",
    "canola oil": "rapeseed oil",
    "chicken breasts": "chicken breast",
    "chicken thighs": "chicken thigh",
    "ground beef": "beef",
    "minced beef": "beef",
    "ground turkey": "turkey",
    "minced turkey": "turkey",
}

DESCRIPTOR_WORDS = {
    "fresh", "frozen", "canned", "dried", "raw", "cooked",
    "chopped", "diced", "minced", "sliced", "grated", "shredded",
    "boneless", "skinless", "lean", "large", "small", "medium",
    "organic", "optional", "plain", "whole", "halved", "crushed",
    "toasted", "softened", "melted", "warm", "cold", "drained",
    "rinsed", "beaten", "packed", "extra", "virgin", "firmly",
    "choice", "chef", "reserve", "reserved", "flavored", "flavoured"
}

UNIT_WORDS = {
    "cup", "cups", "tbsp", "tsp", "teaspoon", "teaspoons",
    "tablespoon", "tablespoons", "oz", "ounce", "ounces",
    "lb", "lbs", "pound", "pounds", "gram", "grams", "g",
    "kg", "ml", "l", "pinch", "clove", "cloves", "slice", "slices",
    "can", "cans", "package", "packages", "pack", "packs",
    "carton", "cartons", "jar", "jars", "inch", "inches"
}


def normalize_ingredient_name(text: str) -> str:
    """Normalize raw ingredient text for matching, scoring, and filtering.

    Parameters
    ----------
    text : str
        Raw ingredient text.

    Returns
    -------
    str
        Normalized ingredient phrase.
    """
    if not isinstance(text, str):
        return ""

    s = text.lower().strip()
    s = re.sub(r"\([^)]*\)", " ", s)
    s = re.sub(r"\b\d+\s*/\s*\d+\b", " ", s)
    s = re.sub(r"\b\d+\b", " ", s)
    s = re.sub(r"[^a-z\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    words = [w for w in s.split() if w not in DESCRIPTOR_WORDS and w not in UNIT_WORDS]
    s = " ".join(words).strip()

    if s in INGREDIENT_ALIASES:
        s = INGREDIENT_ALIASES[s]

    singular_map = {
        "tomatoes": "tomato",
        "potatoes": "potato",
        "onions": "onion",
        "carrots": "carrot",
        "peppers": "pepper",
        "beans": "bean",
        "lentils": "lentil",
        "pecans": "pecan",
        "walnuts": "walnut",
        "almonds": "almond",
        "eggs": "egg",
    }
    s = " ".join(singular_map.get(w, w) for w in s.split()).strip()

    if s in INGREDIENT_ALIASES:
        s = INGREDIENT_ALIASES[s]

    return s


def clean_ingredient(token: str) -> str:
    """Compatibility wrapper around ingredient normalization.

    Parameters
    ----------
    token : str
        Raw ingredient token.

    Returns
    -------
    str
        Normalized ingredient text.
    """
    return normalize_ingredient_name(token)


def normalize_user_ingredients(items):
    """Normalize user pantry, must-include, or must-exclude ingredients.

    Parameters
    ----------
    items : str | list[str] | None
        User-provided ingredient input.

    Returns
    -------
    set[str]
        Normalized ingredient set.
    """
    if items is None:
        return set()

    if isinstance(items, str):
        items = [items]

    normalized = set()
    for item in items:
        norm = normalize_ingredient_name(item)
        if norm:
            normalized.add(norm)
    return normalized

## 2. User Nutrition Target Functions

In [8]:
"""Functions for BMI, BMR, TDEE, calorie adjustments, and meal macro targets."""

def calculate_bmi(weight_kg, height_cm):
    """Compute BMI from weight in kg and height in cm."""
    height_m = height_cm / 100
    return weight_kg / (height_m ** 2)


def calculate_bmr(weight_kg, height_cm, age, sex):
    """Compute BMR using Mifflin-St Jeor style equations."""
    sex = str(sex).lower()
    if sex == "male":
        return (9.99 * weight_kg) + (6.25 * height_cm) - (4.92 * age) + 5
    return (9.99 * weight_kg) + (6.25 * height_cm) - (4.92 * age) - 161


def calculate_tdee(bmr, activity_level):
    """Convert BMR to TDEE using an activity multiplier."""
    activity_multipliers = {
        "sedentary": 1.2,
        "lightly_active": 1.375,
        "moderate": 1.55,
        "very_active": 1.725,
        "extra_active": 1.9,
    }
    return bmr * activity_multipliers.get(activity_level, 1.2)


def adjust_calories(tdee, goal, sex):
    """Adjust calories based on goal with a simple safety floor for weight loss."""
    if goal == "weight_loss":
        target = tdee * 0.8
        floor = 1500 if str(sex).lower() == "male" else 1200
        return max(target, floor)
    if goal == "weight_gain":
        return tdee * 1.1
    return tdee


def calculate_macros(calories, goal, weight_kg):
    """Return daily protein, fat, and carbohydrate gram targets."""
    if goal == "weight_loss":
        protein = 1.8 * weight_kg
        fat = 0.8 * weight_kg
    elif goal == "weight_gain":
        protein = 1.6 * weight_kg
        fat = 0.9 * weight_kg
    else:
        protein = 1.6 * weight_kg
        fat = 0.8 * weight_kg

    carbs = max((calories - protein * 4 - fat * 9) / 4, 0)
    return protein, fat, carbs


def build_user_targets(user_profile, meals_per_day=3):
    """Build per-day and per-meal targets from user profile inputs.

    Parameters
    ----------
    user_profile : dict
        Dictionary with keys such as age, weight, height, sex, activity_level, and goal.
    meals_per_day : int, default=3
        Number of meals per day.

    Returns
    -------
    dict
        Daily and per-meal calorie and macro targets.
    """
    bmi = calculate_bmi(user_profile["weight"], user_profile["height"])
    bmr = calculate_bmr(
        user_profile["weight"],
        user_profile["height"],
        user_profile["age"],
        user_profile["sex"],
    )
    tdee = calculate_tdee(bmr, user_profile["activity_level"])
    target_calories = adjust_calories(tdee, user_profile["goal"], user_profile["sex"])
    protein_target, fat_target, carb_target = calculate_macros(
        target_calories,
        user_profile["goal"],
        user_profile["weight"],
    )

    return {
        "bmi": bmi,
        "bmr": bmr,
        "tdee": tdee,
        "target_calories": target_calories,
        "target_protein": protein_target,
        "target_fat": fat_target,
        "target_carbs": carb_target,
        "meal_calories": target_calories / meals_per_day,
        "meal_protein": protein_target / meals_per_day,
        "meal_fat": fat_target / meals_per_day,
        "meal_carbs": carb_target / meals_per_day,
    }

## 3. Recipe Feature Engineering

In [9]:
"""Feature engineering for food tags, weight-loss features, Nutri-Score, NRF score, and health flags."""

def infer_food_tags(row):
    """Infer food-type tags from normalized ingredients, category, and keywords.

    Returns tags such as:
    - vegan
    - vegetarian
    - meat
    - poultry
    - chicken
    - seafood
    - dairy
    - egg
    - dessert
    - salad
    - breakfast
    """
    tags = set()

    ingredients = row.get("ingredients_clean", [])
    ingredients = [str(x).lower() for x in ingredients] if isinstance(ingredients, list) else []

    category = str(row.get("RecipeCategory", "")).lower()
    keywords = row.get("Keywords", [])
    keywords = [str(x).lower() for x in keywords] if isinstance(keywords, list) else []

    text = " ".join(ingredients + [category] + keywords)

    meat_terms = {"beef", "pork", "lamb", "bacon", "ham", "sausage"}
    poultry_terms = {"chicken", "turkey", "duck"}
    seafood_terms = {"fish", "salmon", "tuna", "shrimp", "prawn", "cod", "crab"}
    dairy_terms = {"milk", "cheese", "yogurt", "butter", "cream"}
    egg_terms = {"egg"}
    vegan_blockers = meat_terms | poultry_terms | seafood_terms | dairy_terms | egg_terms

    if any(t in text for t in poultry_terms):
        tags.update({"chicken", "poultry"})
    if any(t in text for t in seafood_terms):
        tags.add("seafood")
    if any(t in text for t in meat_terms):
        tags.add("meat")
    if any(t in text for t in dairy_terms):
        tags.add("dairy")
    if any(t in text for t in egg_terms):
        tags.add("egg")

    if not any(t in text for t in meat_terms | poultry_terms | seafood_terms):
        tags.add("vegetarian")

    if not any(t in text for t in vegan_blockers):
        tags.add("vegan")

    if "dessert" in category:
        tags.add("dessert")
    if "salad" in category:
        tags.add("salad")
    if "breakfast" in category:
        tags.add("breakfast")

    return sorted(tags)


def add_weight_loss_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create per-100g, density, ratio, and macro-label features used in weight-loss scoring.

    Parameters
    ----------
    df : pandas.DataFrame
        Recipe dataframe with nutrition and serving columns.

    Returns
    -------
    pandas.DataFrame
        Dataframe with added nutrition-engineered features.
    """
    df = df.copy()

    numeric_cols = [
        "calories", "protein", "fat", "sat_fat", "carbs",
        "sugar", "fiber", "sodium", "serving_g", "cal_per_g", "cook_time"
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    needed = [
        "calories", "protein", "fat", "sat_fat", "carbs",
        "sugar", "fiber", "sodium", "serving_g", "cal_per_g"
    ]
    df = df.dropna(subset=[c for c in needed if c in df.columns]).copy()
    df = df[(df["calories"] > 0) & (df["serving_g"] > 0) & (df["cal_per_g"] > 0)].copy()

    df["energy_kcal_100g"] = df["cal_per_g"] * 100
    df["protein_100g"] = df["protein"] / (df["serving_g"] + EPS) * 100
    df["fat_100g"] = df["fat"] / (df["serving_g"] + EPS) * 100
    df["sat_fat_100g"] = df["sat_fat"] / (df["serving_g"] + EPS) * 100
    df["carbs_100g"] = df["carbs"] / (df["serving_g"] + EPS) * 100
    df["sugar_100g"] = df["sugar"] / (df["serving_g"] + EPS) * 100
    df["fiber_100g"] = df["fiber"] / (df["serving_g"] + EPS) * 100
    df["sodium_100g"] = df["sodium"] / (df["serving_g"] + EPS) * 100

    df["protein_per_kcal"] = df["protein"] / (df["calories"] + EPS)
    df["fiber_per_kcal"] = df["fiber"] / (df["calories"] + EPS)
    df["sugar_per_kcal"] = df["sugar"] / (df["calories"] + EPS)

    df["low_calorie_density"] = (df["cal_per_g"] <= 1.5).astype(int)
    df["high_protein_flag"] = (df["protein_100g"] >= 8).astype(int)
    df["high_fiber_flag"] = (df["fiber_100g"] >= 3).astype(int)
    df["high_sugar_flag"] = (df["sugar_100g"] >= 10).astype(int)
    df["high_sat_fat_flag"] = (df["sat_fat_100g"] >= 5).astype(int)
    df["high_sodium_flag"] = (df["sodium_100g"] >= 600).astype(int)

    df["protein_kcal"] = df["protein"] * 4
    df["carb_kcal"] = df["carbs"] * 4
    df["fat_kcal"] = df["fat"] * 9

    df["protein_pct"] = df["protein_kcal"] / (df["calories"] + EPS)
    df["carb_pct"] = df["carb_kcal"] / (df["calories"] + EPS)
    df["fat_pct"] = df["fat_kcal"] / (df["calories"] + EPS)

    def classify_protein(p):
        if pd.isna(p):
            return np.nan
        if p < 0.12:
            return "low"
        if p < 0.20:
            return "moderate"
        return "high"

    def classify_carb(p):
        if pd.isna(p):
            return np.nan
        if p < 0.45:
            return "low"
        if p <= 0.65:
            return "moderate"
        return "high"

    def classify_fat(p):
        if pd.isna(p):
            return np.nan
        if p < 0.20:
            return "low"
        if p <= 0.35:
            return "moderate"
        return "high"

    df["protein_level"] = df["protein_pct"].apply(classify_protein)
    df["carb_level"] = df["carb_pct"].apply(classify_carb)
    df["fat_level"] = df["fat_pct"].apply(classify_fat)

    df["macro_labels"] = df.apply(
        lambda r: [
            lbl for lbl, ok in [
                ("High Protein", r["protein_level"] == "high"),
                ("Fiber Friendly", r["fiber_100g"] >= 3),
                ("Low Sugar", r["sugar_100g"] <= 5),
                ("Low Sodium", r["sodium_100g"] <= 120),
                ("Low Calorie Density", r["low_calorie_density"] == 1),
            ] if ok
        ],
        axis=1,
    )

    return df


def add_weight_loss_score(df: pd.DataFrame) -> pd.DataFrame:
    """Compute a heuristic weight-loss suitability score for each recipe."""
    df = df.copy()

    pos = (
        2.5 * df["protein_per_kcal"].fillna(0) +
        2.0 * df["fiber_per_kcal"].fillna(0) +
        1.5 * df["low_calorie_density"].fillna(0) +
        1.0 * df["high_protein_flag"].fillna(0) +
        1.0 * df["high_fiber_flag"].fillna(0)
    )

    neg = (
        1.8 * df["high_sugar_flag"].fillna(0) +
        1.5 * df["high_sat_fat_flag"].fillna(0) +
        1.2 * df["high_sodium_flag"].fillna(0)
    )

    cook_penalty = np.where(df.get("cook_time", pd.Series(0, index=df.index)).fillna(0) > 60, 0.25, 0.0)

    df["weight_loss_score"] = (pos - neg - cook_penalty).round(4)
    return df


def add_weight_loss_label(df: pd.DataFrame, quantile=0.60) -> pd.DataFrame:
    """Create binary and text labels from the weight-loss score.

    Parameters
    ----------
    df : pandas.DataFrame
        Recipe dataframe with a weight_loss_score column.
    quantile : float, default=0.60
        Quantile threshold used to define the positive class.

    Returns
    -------
    pandas.DataFrame
        Dataframe with weight_loss_friendly and weight_loss_priority columns.
    """
    df = df.copy()
    threshold = df["weight_loss_score"].quantile(quantile)
    df["weight_loss_friendly"] = (df["weight_loss_score"] >= threshold).astype(int)
    df["weight_loss_priority"] = np.where(
        df["weight_loss_friendly"] == 1, "recommended", "less_recommended"
    )
    return df


FVP_WORDS = {
    "apple", "banana", "orange", "lemon", "lime", "grape", "mango", "papaya",
    "pineapple", "peach", "pear", "plum", "cherry", "berry", "blueberry",
    "strawberry", "raspberry", "blackberry", "apricot", "fig", "date",
    "raisin", "coconut", "avocado", "tomato", "carrot", "spinach", "kale",
    "lettuce", "cabbage", "broccoli", "cauliflower", "zucchini", "eggplant",
    "bell pepper", "pepper", "onion", "garlic", "ginger", "celery", "cucumber",
    "pumpkin", "squash", "yam", "sweet potato", "potato", "corn", "mushroom",
    "pea", "green bean", "okra", "beet", "radish", "turnip", "artichoke"
}

PULSE_WORDS = {
    "lentil", "chickpea", "black bean", "kidney bean", "white bean",
    "cannellini bean", "pinto bean", "split pea", "pea", "bean"
}

NUT_WORDS = {
    "almond", "walnut", "pecan", "cashew", "hazelnut", "pistachio",
    "macadamia", "peanut", "nut"
}

HEALTHY_OIL_WORDS = {"olive oil", "walnut oil", "rapeseed oil", "canola oil"}


def add_nutriscore_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add ingredient-group proxy features used for approximate Nutri-Score.

    Because exact ingredient-by-weight percentages are unavailable, fruits,
    vegetables, pulses, nuts, and healthy oils are estimated using counts and ratios.
    """
    df = df.copy()

    def count_groups(ings):
        if not isinstance(ings, list):
            ings = []

        fvp = sum(any(term in ing for term in FVP_WORDS) for ing in ings)
        pulses = sum(any(term in ing for term in PULSE_WORDS) for ing in ings)
        nuts = sum(any(term in ing for term in NUT_WORDS) for ing in ings)
        healthy_oils = sum(any(term in ing for term in HEALTHY_OIL_WORDS) for ing in ings)

        total = max(len(ings), 1)
        return pd.Series({
            "fvp_count": fvp,
            "pulse_count": pulses,
            "nut_count": nuts,
            "healthy_oil_count": healthy_oils,
            "fvp_ratio": fvp / total,
            "pulse_ratio": pulses / total,
            "nut_ratio": nuts / total,
            "healthy_oil_ratio": healthy_oils / total,
        })

    counts = df["ingredients_clean"].apply(count_groups)
    df = pd.concat([df, counts], axis=1)
    df["beneficial_ingredient_ratio"] = (
        df["fvp_ratio"] + df["pulse_ratio"] + df["nut_ratio"] + df["healthy_oil_ratio"]
    ).clip(upper=1.0)
    return df


def _points_from_thresholds(value, thresholds):
    """Assign points based on ordered thresholds."""
    for i, t in enumerate(thresholds):
        if value <= t:
            return i
    return len(thresholds)


def add_nutriscore_points(df: pd.DataFrame) -> pd.DataFrame:
    """Compute approximate Nutri-Score numeric points and A-to-E labels.

    Notes
    -----
    This is a recipe-oriented approximation. FVPN percentage is estimated using
    ingredient ratios instead of exact weight shares.
    """
    df = df.copy()

    df["energy_kj_100g"] = df["energy_kcal_100g"] * 4.184

    df["ns_energy_points"] = df["energy_kj_100g"].apply(
        lambda x: _points_from_thresholds(x, [335, 670, 1005, 1340, 1675, 2010, 2345, 2680, 3015, 3350])
    )
    df["ns_sugar_points"] = df["sugar_100g"].apply(
        lambda x: _points_from_thresholds(x, [4.5, 9, 13.5, 18, 22.5, 27, 31, 36, 40, 45])
    )
    df["ns_satfat_points"] = df["sat_fat_100g"].apply(
        lambda x: _points_from_thresholds(x, [1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
    )
    df["ns_sodium_points"] = df["sodium_100g"].apply(
        lambda x: _points_from_thresholds(x, [90, 180, 270, 360, 450, 540, 630, 720, 810, 900])
    )

    df["ns_negative_points"] = (
        df["ns_energy_points"] +
        df["ns_sugar_points"] +
        df["ns_satfat_points"] +
        df["ns_sodium_points"]
    )

    def fvpn_points(r):
        if r >= 0.80:
            return 5
        if r >= 0.60:
            return 2
        if r >= 0.40:
            return 1
        return 0

    def fiber_points(x):
        return _points_from_thresholds(x, [0.9, 1.9, 2.8, 3.7, 4.7])

    def protein_points(x):
        return _points_from_thresholds(x, [1.6, 3.2, 4.8, 6.4, 8.0])

    df["ns_fvpn_points"] = df["beneficial_ingredient_ratio"].apply(fvpn_points)
    df["ns_fiber_points"] = df["fiber_100g"].apply(fiber_points)
    df["ns_protein_points"] = df["protein_100g"].apply(protein_points)

    df["nutri_score_numeric"] = (
        df["ns_negative_points"] -
        df["ns_fvpn_points"] -
        df["ns_fiber_points"] -
        df["ns_protein_points"]
    )

    def label_food(score):
        if score <= -1:
            return "A"
        if score <= 2:
            return "B"
        if score <= 10:
            return "C"
        if score <= 18:
            return "D"
        return "E"

    df["nutri_score_label"] = df["nutri_score_numeric"].apply(label_food)
    return df


def compute_nrf23_health_score(df: pd.DataFrame) -> pd.DataFrame:
    """Compute a simple NRF-style health score from beneficial and limiting nutrients."""
    df = df.copy()

    required_cols = ["protein", "fiber", "sugar", "sat_fat", "sodium"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    for col in required_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["protein"] = df["protein"].clip(lower=0, upper=150)
    df["fiber"] = df["fiber"].clip(lower=0, upper=60)
    df["sugar"] = df["sugar"].clip(lower=0, upper=200)
    df["sat_fat"] = df["sat_fat"].clip(lower=0, upper=80)
    df["sodium"] = df["sodium"].clip(lower=0, upper=8000)

    protein_ref = 50.0
    fiber_ref = 25.0
    sugar_max = 50.0
    satfat_max = 20.0
    sodium_max = 2400.0

    df["protein_score_nrf"] = df["protein"] / protein_ref
    df["fiber_score_nrf"] = df["fiber"] / fiber_ref
    df["sugar_penalty_nrf"] = df["sugar"] / sugar_max
    df["satfat_penalty_nrf"] = df["sat_fat"] / satfat_max
    df["sodium_penalty_nrf"] = df["sodium"] / sodium_max

    df["nrf23_raw"] = (
        df["protein_score_nrf"] +
        df["fiber_score_nrf"] -
        df["sugar_penalty_nrf"] -
        df["satfat_penalty_nrf"] -
        df["sodium_penalty_nrf"]
    )

    min_val = df["nrf23_raw"].min()
    max_val = df["nrf23_raw"].max()

    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        df["health_score"] = 50.0
    else:
        df["health_score"] = ((df["nrf23_raw"] - min_val) / (max_val - min_val)) * 100

    df["health_score"] = df["health_score"].round(2)

    q33 = df["nrf23_raw"].quantile(0.33)
    q66 = df["nrf23_raw"].quantile(0.66)

    def classify(x):
        if pd.isna(x):
            return np.nan
        if x >= q66:
            return "Healthy"
        if x >= q33:
            return "Moderate"
        return "Unhealthy"

    df["health_class"] = df["nrf23_raw"].apply(classify)
    return df


def add_medical_risk_flags(df: pd.DataFrame) -> pd.DataFrame:
    """Add basic medical risk flags for common conditions such as diabetes and hypertension."""
    df = df.copy()

    for col in ["sugar", "sodium", "sat_fat", "cholest", "protein", "carbs"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df["risk_diabetes"] = (df["sugar"] > 15).astype(int)
    df["risk_hypertension"] = (df["sodium"] > 600).astype(int)
    df["risk_heart_disease"] = (df["sat_fat"] > 5).astype(int)
    df["risk_cholesterol"] = (df["cholest"] > 100).astype(int)
    df["risk_kidney"] = (df["protein"] > 40).astype(int)
    df["risk_keto_violation"] = (df["carbs"] > 30).astype(int)

    risk_cols = [
        "risk_diabetes",
        "risk_hypertension",
        "risk_heart_disease",
        "risk_cholesterol",
        "risk_kidney",
        "risk_keto_violation",
    ]
    df["medical_risk_score"] = df[risk_cols].sum(axis=1)

    def classify_medical_risk(score):
        if score == 0:
            return "low"
        if score <= 2:
            return "moderate"
        return "high"

    df["medical_risk_level"] = df["medical_risk_score"].apply(classify_medical_risk)

    def risk_reason(row):
        reasons = []
        if row["risk_diabetes"]:
            reasons.append("high sugar")
        if row["risk_hypertension"]:
            reasons.append("high sodium")
        if row["risk_heart_disease"]:
            reasons.append("high saturated fat")
        if row["risk_cholesterol"]:
            reasons.append("high cholesterol")
        if row["risk_kidney"]:
            reasons.append("very high protein")
        if row["risk_keto_violation"]:
            reasons.append("high carbohydrates")
        return ", ".join(reasons) if reasons else "no major medical risk"

    df["medical_risk_reason"] = df.apply(risk_reason, axis=1)
    return df


def filter_recipes_by_health(df: pd.DataFrame, user_conditions=None, strict=True) -> pd.DataFrame:
    """Filter recipes by health-condition risk flags.

    If a required risk column is missing, that condition is skipped instead of raising an error.
    """
    filtered = df.copy()
    user_conditions = {str(x).strip().lower() for x in (user_conditions or [])}

    if not user_conditions:
        return filtered

    condition_to_col = {
        "diabetes": "risk_diabetes",
        "hypertension": "risk_hypertension",
        "heart disease": "risk_heart_disease",
        "cholesterol": "risk_cholesterol",
        "kidney": "risk_kidney",
    }

    for condition, col in condition_to_col.items():
        if condition in user_conditions and strict and col in filtered.columns:
            filtered = filtered[filtered[col] == 0]

    return filtered

## 4. Main Loader

In [ ]:
"""Load the merged recipe CSV, standardize fields, and add app-ready features."""

def load_and_process_recipe_csv(path_csv: str) -> pd.DataFrame:
    """Load and preprocess the merged recipe CSV for modeling and app serving.

    Parameters
    ----------
    path_csv : str
        Path to the merged recipe dataset.

    Returns
    -------
    pandas.DataFrame
        Processed dataframe with recipe display, filtering, and scoring features.
    """
    df = pd.read_csv(path_csv)

    required_cols = [
        "id",
        "name_final",
        "ingredients",
        "steps",
        "Calories",
        "FatContent",
        "SaturatedFatContent",
        "CarbohydrateContent",
        "FiberContent",
        "SugarContent",
        "ProteinContent",
        "CholesterolContent",
        "SodiumContent",
        "servings",
        "serving_g",
        "cal_per_g",
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df["recipe_id"] = df["id"]
    df["final_name"] = df["name_final"].fillna("").astype(str).str.strip()
    df["name"] = df["final_name"]

    if "description_final" in df.columns:
        df["final_description"] = df["description_final"].fillna("").astype(str).str.strip()
    else:
        df["final_description"] = ""

    # ingredients / instructions
    df["ingredients_list"] = df["ingredients"].apply(_safe_list)

    if "ingredients_raw" in df.columns:
        df["ingredients_raw_list"] = df["ingredients_raw"].apply(_safe_list)
    else:
        df["ingredients_raw_list"] = df["ingredients_list"]

    # optional quantities if available from source tables
    if "RecipeIngredientQuantities" in df.columns:
        df["ingredient_quantities"] = df["RecipeIngredientQuantities"].apply(_safe_list)
    else:
        df["ingredient_quantities"] = [[] for _ in range(len(df))]

    df["ingredients_normalized"] = df["ingredients_list"].apply(
        lambda lst: [normalize_ingredient_name(x) for x in lst if normalize_ingredient_name(x)]
    )

    df["ingredients_clean"] = df["ingredients_normalized"].apply(
        lambda lst: sorted(list(set(lst))) if isinstance(lst, list) else []
    )

    df["ingredients_text"] = df["ingredients_clean"].apply(
        lambda lst: " ".join(lst) if isinstance(lst, list) else ""
    )

    df["instructions_list"] = df["steps"].apply(parse_instructions)

    if "Keywords" in df.columns:
        df["Keywords"] = df["Keywords"].apply(_safe_list)
    else:
        df["Keywords"] = [[] for _ in range(len(df))]

    if "RecipeCategory" not in df.columns:
        df["RecipeCategory"] = ""

    if "TotalTime" in df.columns:
        df["cook_time"] = df["TotalTime"].apply(parse_time_to_minutes)
    else:
        df["cook_time"] = np.nan

    df = df.rename(columns={
        "Calories": "calories",
        "FatContent": "fat",
        "SaturatedFatContent": "sat_fat",
        "CarbohydrateContent": "carbs",
        "FiberContent": "fiber",
        "SugarContent": "sugar",
        "ProteinContent": "protein",
        "CholesterolContent": "cholest",
        "SodiumContent": "sodium",
    })

    numeric_cols = [
        "calories", "fat", "sat_fat", "carbs", "fiber", "sugar",
        "protein", "cholest", "sodium", "servings", "serving_g", "cal_per_g", "cook_time"
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=[
        "final_name", "calories", "fat", "sat_fat", "carbs",
        "fiber", "sugar", "protein", "sodium", "serving_g", "cal_per_g"
    ]).copy()

    df = df[
        (df["final_name"] != "") &
        (df["calories"] > 0) &
        (df["serving_g"] > 0) &
        (df["cal_per_g"] > 0)
    ].copy()

    df["food_tags"] = df.apply(infer_food_tags, axis=1)
    df = add_weight_loss_features(df)
    df = add_weight_loss_score(df)
    df = add_weight_loss_label(df, quantile=0.60)
    df = add_weight_loss_rank_features(df)
    df = add_nutriscore_features(df)
    df = add_nutriscore_points(df)
    df = compute_nrf23_health_score(df)
    df = add_medical_risk_flags(df)

    df = df.drop_duplicates(subset=["final_name", "ingredients_text"]).reset_index(drop=True)
    return df

## 5. Ingredient Matching, Macro Scoring, and Recommendation Utilities

In [15]:
"""Functions for pantry matching, must-include/exclude checks, macro matching, and final ranking."""

def ingredient_overlap(recipe_ingredients, user_ingredients):
    """Return matched and missing ingredients between a recipe and user pantry."""
    recipe_set = set(recipe_ingredients) if isinstance(recipe_ingredients, list) else set()
    user_set = set(user_ingredients) if isinstance(user_ingredients, (set, list)) else set()

    matched = sorted(recipe_set & set(user_set))
    missing = sorted(recipe_set - set(user_set))
    coverage = len(matched) / max(len(recipe_set), 1)

    return {
        "matched_ingredients": matched,
        "missing_ingredients": missing,
        "matched_count": len(matched),
        "missing_count": len(missing),
        "ingredient_coverage": coverage,
    }


def recipe_passes_ingredient_rules(recipe_ingredients, must_include=None, must_exclude=None):
    """Check whether a recipe satisfies must-include and must-exclude rules."""
    recipe_set = set(recipe_ingredients) if isinstance(recipe_ingredients, list) else set()
    must_include = normalize_user_ingredients(must_include)
    must_exclude = normalize_user_ingredients(must_exclude)

    include_ok = must_include.issubset(recipe_set) if must_include else True
    exclude_ok = recipe_set.isdisjoint(must_exclude) if must_exclude else True
    return include_ok and exclude_ok


def compute_ingredient_score(recipe_ingredients, pantry_ingredients=None, must_include=None, must_exclude=None):
    """Compute an ingredient-matching score used for ranking recommendations."""
    recipe_set = set(recipe_ingredients) if isinstance(recipe_ingredients, list) else set()
    pantry_set = normalize_user_ingredients(pantry_ingredients)
    include_set = normalize_user_ingredients(must_include)
    exclude_set = normalize_user_ingredients(must_exclude)

    matched = recipe_set & pantry_set
    include_hits = recipe_set & include_set
    exclude_hits = recipe_set & exclude_set

    coverage = len(matched) / max(len(recipe_set), 1)
    include_bonus = len(include_hits) / max(len(include_set), 1) if include_set else 0
    exclude_penalty = len(exclude_hits) / max(len(exclude_set), 1) if exclude_set else 0

    score = 0.7 * coverage + 0.3 * include_bonus - 1.0 * exclude_penalty
    return max(0.0, round(score, 4))


def add_user_ingredient_features(df, pantry_ingredients=None, must_include=None, must_exclude=None):
    """Add matched, missing, coverage, and ingredient_score columns to the dataframe."""
    df = df.copy()

    pantry_set = normalize_user_ingredients(pantry_ingredients)
    include_set = normalize_user_ingredients(must_include)
    exclude_set = normalize_user_ingredients(must_exclude)

    def process_row(ings):
        result = ingredient_overlap(ings, pantry_set)
        result["ingredient_score"] = compute_ingredient_score(
            ings,
            pantry_ingredients=pantry_set,
            must_include=include_set,
            must_exclude=exclude_set,
        )
        result["passes_include_exclude"] = recipe_passes_ingredient_rules(
            ings,
            must_include=include_set,
            must_exclude=exclude_set,
        )
        return pd.Series(result)

    features = df["ingredients_clean"].apply(process_row)
    df = pd.concat([df, features], axis=1)
    return df


def compute_macro_score(df: pd.DataFrame, user_targets: dict) -> pd.DataFrame:
    """Compute recipe-to-user macro alignment scores.

    The score rewards closeness to per-meal calorie, protein, fat, and carbohydrate targets.
    Higher values indicate better alignment.
    """
    df = df.copy()

    meal_cal = max(user_targets.get("meal_calories", 0), EPS)
    meal_pro = max(user_targets.get("meal_protein", 0), EPS)
    meal_fat = max(user_targets.get("meal_fat", 0), EPS)
    meal_carbs = max(user_targets.get("meal_carbs", 0), EPS)

    cal_fit = 1 - (df["calories"] - meal_cal).abs() / meal_cal
    pro_fit = 1 - (df["protein"] - meal_pro).abs() / meal_pro
    fat_fit = 1 - (df["fat"] - meal_fat).abs() / meal_fat
    carb_fit = 1 - (df["carbs"] - meal_carbs).abs() / meal_carbs

    cal_fit = cal_fit.clip(lower=0)
    pro_fit = pro_fit.clip(lower=0)
    fat_fit = fat_fit.clip(lower=0)
    carb_fit = carb_fit.clip(lower=0)

    df["macro_score"] = (
        0.35 * cal_fit +
        0.25 * pro_fit +
        0.20 * fat_fit +
        0.20 * carb_fit
    ) * 100

    df["macro_score"] = df["macro_score"].round(2)
    return df


def apply_preference_filters(
    df,
    preferred_food_types=None,
    max_cook_time=None,
    protein_pref=None,
    fat_pref=None,
    carb_pref=None,
):
    """Apply user food-type, cook-time, and macro-level preferences to the recipe table."""
    out = df.copy()

    if preferred_food_types:
        pref = {str(x).strip().lower() for x in preferred_food_types}
        out = out[out["food_tags"].apply(lambda tags: bool(set(tags).intersection(pref)))]

    if max_cook_time is not None and "cook_time" in out.columns:
        out = out[out["cook_time"].fillna(np.inf) <= max_cook_time]

    if protein_pref:
        out = out[out["protein_level"] == str(protein_pref).lower()]
    if fat_pref:
        out = out[out["fat_level"] == str(fat_pref).lower()]
    if carb_pref:
        out = out[out["carb_level"] == str(carb_pref).lower()]

    return out.reset_index(drop=True)


def sort_recommendations(df, sort_by="weight_loss_probability", ascending=False):
    """Sort recommendations by one of the supported ranking columns."""
    valid_cols = {
        "macro_score",
        "ingredient_score",
        "weight_loss_score",
        "weight_loss_probability",
        "health_score",
    }

    if sort_by not in valid_cols or sort_by not in df.columns:
        sort_by = "weight_loss_probability"

    return df.sort_values(by=sort_by, ascending=ascending).reset_index(drop=True)


def build_recommendations(
    df,
    user_profile,
    pantry_ingredients=None,
    must_include=None,
    must_exclude=None,
    preferred_food_types=None,
    max_cook_time=None,
    meals_per_day=3,
    n_recommendations=10,
    protein_pref=None,
    fat_pref=None,
    carb_pref=None,
    health_issues=None,
    sort_by="weight_loss_probability",
):
    """Create the final recommendation table for the app.

    This function combines:
    - user calorie and macro targets
    - health-condition filtering
    - food-type and cook-time filtering
    - ingredient matching
    - macro score
    - final ranking

    Returns
    -------
    pandas.DataFrame
        Ranked recommendations ready for app display.
    """
    user_targets = build_user_targets(user_profile, meals_per_day=meals_per_day)

    out = df.copy()
    out = filter_recipes_by_health(out, user_conditions=health_issues, strict=True)
    out = apply_preference_filters(
        out,
        preferred_food_types=preferred_food_types,
        max_cook_time=max_cook_time,
        protein_pref=protein_pref,
        fat_pref=fat_pref,
        carb_pref=carb_pref,
    )
    out = add_user_ingredient_features(
        out,
        pantry_ingredients=pantry_ingredients,
        must_include=must_include,
        must_exclude=must_exclude,
    )
    out = out[out["passes_include_exclude"] == True].copy()
    out = compute_macro_score(out, user_targets=user_targets)
    out = sort_recommendations(out, sort_by=sort_by, ascending=False)

    display_cols = [
        "recipe_id", "name", "final_description", "nutri_score_label", "macro_labels",
        "macro_score", "ingredient_score", "weight_loss_score", "weight_loss_probability",
        "health_score", "matched_ingredients", "missing_ingredients", "ingredients_list",
        "instructions_list", "food_tags", "cook_time", "calories", "protein", "fat", "carbs",
    ]
    display_cols = [c for c in display_cols if c in out.columns]
    return out[display_cols].head(n_recommendations)

## 6. Weight-Loss rank

In [23]:
def add_weight_loss_rank_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add rank-based weight-loss fields without training a model.

    This replaces the previous ML probability approach because the old model
    was trained on labels derived from the same engineered nutrition features,
    which created near-perfect but not very meaningful predictions.

    Added columns:
        - weight_loss_score_pct:
            Percentile rank of weight_loss_score in [0, 1].
        - weight_loss_probability:
            A monotonic score-based proxy in [0, 1] kept for UI compatibility.
            This is NOT a learned probability.
        - weight_loss_band:
            Human-readable bucket for display and sorting.
    """
    df = df.copy()

    if "weight_loss_score" not in df.columns:
        raise ValueError("Missing required column: 'weight_loss_score'")

    score = pd.to_numeric(df["weight_loss_score"], errors="coerce")

    # Percentile rank: robust and easy to interpret
    df["weight_loss_score_pct"] = score.rank(pct=True, method="average")

    # Smooth proxy in [0,1] based on z-score then sigmoid
    mean_ = score.mean()
    std_ = score.std(ddof=0)

    if pd.isna(std_) or std_ == 0:
        z = pd.Series(np.zeros(len(df)), index=df.index)
    else:
        z = (score - mean_) / std_

    df["weight_loss_probability"] = 1 / (1 + np.exp(-z))
    df["weight_loss_probability"] = df["weight_loss_probability"].clip(0, 1)

    def to_band(p):
        if pd.isna(p):
            return np.nan
        if p >= 0.80:
            return "very_high"
        elif p >= 0.60:
            return "high"
        elif p >= 0.40:
            return "moderate"
        elif p >= 0.20:
            return "low"
        return "very_low"

    df["weight_loss_band"] = df["weight_loss_score_pct"].apply(to_band)

    return df

## 7. Ingredient Similarity Search

In [24]:
"""Build and use ingredient-text TF-IDF features for content-based similarity search."""

def fit_ingredient_tfidf(df: pd.DataFrame, vectorizer_path: str = TFIDF_PATH, matrix_path: str = INGREDIENT_MATRIX_PATH):
    """Fit a TF-IDF model on normalized ingredient text and save both vectorizer and matrix."""
    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=30000,
    )
    matrix = vectorizer.fit_transform(df["ingredients_text"].fillna(""))
    joblib.dump(vectorizer, vectorizer_path)
    save_npz(matrix_path, matrix)
    print(f"Saved TF-IDF vectorizer -> {vectorizer_path}")
    print(f"Saved ingredient matrix -> {matrix_path}")
    return vectorizer, matrix


def load_ingredient_tfidf(vectorizer_path: str = TFIDF_PATH, matrix_path: str = INGREDIENT_MATRIX_PATH):
    """Load the saved ingredient TF-IDF vectorizer and sparse matrix."""
    vectorizer = joblib.load(vectorizer_path)
    matrix = load_npz(matrix_path)
    return vectorizer, matrix


def similar_recipes_by_ingredients(df: pd.DataFrame, recipe_name: str, top_k: int = 10, vectorizer=None, matrix=None):
    """Return recipes with similar ingredient profiles to a given recipe name."""
    if vectorizer is None or matrix is None:
        vectorizer, matrix = load_ingredient_tfidf()

    name_series = df["name"].fillna("").str.lower()
    idx_matches = df.index[name_series == str(recipe_name).lower()].tolist()
    if not idx_matches:
        raise ValueError(f"Recipe not found: {recipe_name}")

    idx = idx_matches[0]
    sims = cosine_similarity(matrix[idx], matrix).ravel()
    order = np.argsort(-sims)

    out = df.iloc[order].copy()
    out["ingredient_similarity"] = sims[order]
    out = out[out.index != idx]

    cols = [c for c in ["name", "ingredient_similarity", "nutri_score_label", "health_score", "weight_loss_probability"] if c in out.columns]
    return out[cols].head(top_k)

## 8. Save and Load Processed Files Safely

To avoid VS Code becoming unstable with one huge object-heavy dataframe:
- save a **model-ready parquet** with mostly numeric and text columns
- save a **display-ready parquet** with list columns serialized as JSON strings
- load only the model-ready file during training and most analysis

In [11]:
"""Save and load processed recipe data using separate model and display files."""

LIST_LIKE_COLUMNS = [
    "ingredients_list",
    "ingredients_raw_list",
    "ingredient_quantities",
    "ingredients_normalized",
    "ingredients_clean",
    "instructions_list",
    "food_tags",
    "Keywords",
    "macro_labels",
    "matched_ingredients",
    "missing_ingredients",
    "ingredient_rows",
]

DISPLAY_COLUMNS = [
    "recipe_id",
    "name",
    "final_name",
    "final_description",

    # full recipe output
    "ingredients_list",
    "ingredients_raw_list",
    "ingredient_quantities",
    "ingredients_clean",
    "ingredient_rows",
    "instructions_list",

    # display tags / metadata
    "food_tags",
    "cook_time",
    "servings",
    "serving_g",

    # nutrition shown to user
    "calories",
    "protein",
    "fat",
    "carbs",
    "fiber",
    "sugar",
    "sodium",

    # scores shown to user
    "weight_loss_score",
    "weight_loss_friendly",
    "weight_loss_priority",
    "weight_loss_score_pct",
    "weight_loss_probability",
    "weight_loss_band",
    "health_score",
    "health_class",
    "nutri_score_numeric",
    "nutri_score_label",
    "medical_risk_score",
    "medical_risk_level",
    "medical_risk_reason",
    "protein_level",
    "fat_level",
    "carb_level",
    "macro_labels",
]

MODEL_COLUMNS = [
    "recipe_id",
    "name",
    "final_name",
    "ingredients_text",
    "cook_time",
    "servings",
    "serving_g",
    "calories",
    "protein",
    "fat",
    "sat_fat",
    "carbs",
    "fiber",
    "sugar",
    "cholest",
    "sodium",
    "cal_per_g",
    "energy_kcal_100g",
    "energy_kj_100g",
    "protein_100g",
    "fat_100g",
    "sat_fat_100g",
    "carbs_100g",
    "fiber_100g",
    "sugar_100g",
    "sodium_100g",
    "protein_per_kcal",
    "fiber_per_kcal",
    "sugar_per_kcal",
    "protein_pct",
    "fat_pct",
    "carb_pct",
    "protein_level",
    "fat_level",
    "carb_level",
    "low_calorie_density",
    "high_protein_flag",
    "high_fiber_flag",
    "high_sugar_flag",
    "high_sat_fat_flag",
    "high_sodium_flag",
    "weight_loss_score",
    "weight_loss_friendly",
    "weight_loss_priority",
    "weight_loss_score_pct",
    "weight_loss_probability",
    "weight_loss_band",
    "fvp_count",
    "pulse_count",
    "nut_count",
    "healthy_oil_count",
    "fvp_ratio",
    "pulse_ratio",
    "nut_ratio",
    "healthy_oil_ratio",
    "beneficial_ingredient_ratio",
    "ns_energy_points",
    "ns_sugar_points",
    "ns_satfat_points",
    "ns_sodium_points",
    "ns_fvpn_points",
    "ns_fiber_points",
    "ns_protein_points",
    "ns_negative_points",
    "nutri_score_numeric",
    "nutri_score_label",
    "nrf23_raw",
    "health_score",
    "health_class",
    "risk_diabetes",
    "risk_hypertension",
    "risk_heart_disease",
    "risk_cholesterol",
    "risk_kidney",
    "risk_keto_violation",
    "medical_risk_score",
    "medical_risk_level",
    "medical_risk_reason",
]

COMPACT_MODEL_COLUMNS = [
    "recipe_id",
    "name",
    "final_name",
    "ingredients_text",
    "cook_time",
    "servings",
    "serving_g",
    "calories",
    "protein",
    "fat",
    "sat_fat",
    "carbs",
    "fiber",
    "sugar",
    "sodium",
    "cal_per_g",
    "energy_kcal_100g",
    "protein_100g",
    "fat_100g",
    "sat_fat_100g",
    "carbs_100g",
    "fiber_100g",
    "sugar_100g",
    "sodium_100g",
    "protein_level",
    "fat_level",
    "carb_level",
    "weight_loss_score",
    "weight_loss_score_pct",
    "weight_loss_probability",
    "weight_loss_band",
    "weight_loss_friendly",
    "health_score",
    "health_class",
    "nutri_score_numeric",
    "nutri_score_label",
    "risk_diabetes",
    "risk_hypertension",
    "risk_heart_disease",
    "risk_cholesterol",
    "risk_kidney",
    "risk_keto_violation",
    "medical_risk_score",
    "medical_risk_level",
    "medical_risk_reason",
]

def _serialize_list_columns(df: pd.DataFrame, cols):
    """Serialize list columns as JSON strings for stable parquet storage."""
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = out[col].apply(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x)
    return out


def _deserialize_list_columns(df: pd.DataFrame, cols):
    """Deserialize JSON-string list columns back into Python lists."""
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = out[col].apply(
                lambda x: json.loads(x) if isinstance(x, str) and x.startswith("[") and x.endswith("]") else x
            )
    return out


COMPACT_MODEL_PATH = "processed_data/recipes_model_compact.parquet"

def optimize_dataframe_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast numeric dtypes to reduce memory usage."""
    out = df.copy()

    for col in out.select_dtypes(include=["int64"]).columns:
        out[col] = pd.to_numeric(out[col], downcast="integer")

    for col in out.select_dtypes(include=["float64"]).columns:
        out[col] = pd.to_numeric(out[col], downcast="float")

    for col in out.select_dtypes(include=["object"]).columns:
        nunique = out[col].nunique(dropna=True)
        total = len(out[col])
        if total > 0 and nunique / total < 0.5:
            try:
                out[col] = out[col].astype("category")
            except Exception:
                pass

    return out


def save_processed_recipe_data(
    df: pd.DataFrame,
    model_path: str = PROCESSED_MODEL_PATH,
    display_path: str = PROCESSED_DISPLAY_PATH,
    compact_model_path: str = COMPACT_MODEL_PATH,
):
    """Save processed recipe data into compact model and display files."""
    model_cols = [c for c in MODEL_COLUMNS if c in df.columns]
    display_cols = [c for c in DISPLAY_COLUMNS if c in df.columns]
    compact_cols = [c for c in COMPACT_MODEL_COLUMNS if c in df.columns]

    df_model = optimize_dataframe_dtypes(df[model_cols].copy())
    df_compact = optimize_dataframe_dtypes(df[compact_cols].copy())
    df_display = df[display_cols].copy()
    df_display = _serialize_list_columns(df_display, LIST_LIKE_COLUMNS)

    df_model.to_parquet(model_path, index=False, compression="zstd")
    df_compact.to_parquet(compact_model_path, index=False, compression="zstd")
    df_display.to_parquet(display_path, index=False, compression="zstd")

    print(f"Saved model file   -> {model_path}")
    print(f"Saved compact file -> {compact_model_path}")
    print(f"Saved display file -> {display_path}")


def load_processed_recipe_data(
    model_path: str = PROCESSED_MODEL_PATH,
    display_path: str = PROCESSED_DISPLAY_PATH,
    include_display: bool = True,
    model_columns: list[str] | None = None,
    display_columns: list[str] | None = None,
):
    """Load processed recipe data safely.

    Args:
        model_path: Path to lightweight model-ready parquet.
        display_path: Path to display-ready parquet.
        include_display: Whether to also load display columns.
        model_columns: Optional subset of model columns to load.
        display_columns: Optional subset of display columns to load.

    Returns:
        pd.DataFrame: Loaded dataframe.
    """
    df_model = pd.read_parquet(model_path, columns=model_columns)

    if not include_display:
        return df_model

    df_display = pd.read_parquet(display_path, columns=display_columns)
    df_display = _deserialize_list_columns(df_display, LIST_LIKE_COLUMNS)

    join_key = "recipe_id" if "recipe_id" in df_model.columns and "recipe_id" in df_display.columns else "final_name"
    merged = df_model.merge(df_display, on=join_key, how="left", suffixes=("", "_display"))
    return merged

## 9. End-to-End Processing

In [ ]:
"""Run the end-to-end processing pipeline, train the weight-loss model, add probabilities, and save files."""

df = load_and_process_recipe_csv(RAW_DATA_PATH)

print("Processed shape:", df.shape)

display(df[[
    c for c in [
        "name",
        "calories",
        "protein",
        "fat",
        "carbs",
        "nutri_score_label",
        "health_score",
        "weight_loss_score",
        "weight_loss_score_pct",
        "weight_loss_probability",
        "weight_loss_band"
    ] if c in df.columns
]].head(5))

print("Processed shape after model:", df.shape)
display(df[[
    c for c in [
        "name", "calories", "protein", "fat", "carbs",
        "nutri_score_label", "health_score", "weight_loss_score", "weight_loss_probability"
    ] if c in df.columns
]].head(5))

save_processed_recipe_data(df)

## 10. Lightweight Loading for Daily Work

Use the model-ready file for most development work. It is much lighter than the full display merge.

In [13]:
"""Load only the lightweight model file for faster work inside notebooks and VS Code."""

def load_compact_recipe_data(
    compact_model_path: str = COMPACT_MODEL_PATH,
    columns: list[str] | None = None,
) -> pd.DataFrame:
    """Load the compact model-ready recipe file."""
    return pd.read_parquet(compact_model_path, columns=columns)

df_model = load_compact_recipe_data(columns=[
    "recipe_id",
    "name",
    "calories",
    "protein",
    "fat",
    "carbs",
    "weight_loss_score",
    "weight_loss_probability",
    "health_score",
    "nutri_score_label",
])

print(df_model.shape)
print(df_model.head(3).to_string())


(486662, 10)
   recipe_id                             name  calories  protein   fat  carbs  weight_loss_score  weight_loss_probability  health_score nutri_score_label
0      71247          Cherry Streusel Cobbler     801.0     12.3  29.1  125.0            -1.7534                 0.255594         60.52                 C
1      76133  Reuben and Swiss Casserole Bake     664.4     31.0  45.3   33.9            -1.5656                 0.275748         60.93                 D
2     503816                 Yam-Pecan Recipe     956.8     10.7  53.2  112.8            -3.5135                 0.115314         54.52                 E


## 11. Full Loading for App Output

Only load the merged model+display table when you need recipe cards, ingredients, and instruction steps.

In [12]:
"""Load the merged model and display files when full app output fields are needed."""

df_full = load_processed_recipe_data(include_display=True)
print(df_full.shape)

preview_cols = [
    c for c in [
        "name", "nutri_score_label", "health_score", "weight_loss_probability",
        "ingredients_list", "instructions_list"
    ] if c in df_full.columns
]
display(df_full[preview_cols].head(2))

(486662, 114)


,name,nutri_score_label,health_score,weight_loss_probability,ingredients_list,instructions_list
0,Cherry Streusel Cobbler,C,60.52,0.255594,"[cherry pie filling, condensed milk, melted ma...","[Preheat oven to 375°F., Spread cherry pie fil..."
1,Reuben and Swiss Casserole Bake,D,60.93,0.275748,"[corned beef chopped, sauerkraut cold water, s...","[Set oven to 350 degrees F., Butter a 9 x 13-i..."


## 12. Demo Recommendation Query

This is a quick test cell for the final app flow.

In [16]:
"""Run a demo recommendation query using sample user inputs."""

user_profile = {
    "age": 30,
    "weight": 78,
    "height": 175,
    "sex": "male",
    "activity_level": "moderate",
    "goal": "weight_loss",
}

demo_recs = build_recommendations(
    df_full,
    user_profile=user_profile,
    pantry_ingredients=["olive oil", "tomato", "onion", "chicken breast", "garlic"],
    must_include=["tomato"],
    must_exclude=["peanut"],
    preferred_food_types=["chicken", "vegetarian"],
    max_cook_time=45,
    meals_per_day=3,
    n_recommendations=5,
    protein_pref="high",
    fat_pref=None,
    carb_pref="moderate",
    health_issues=["diabetes"],
    sort_by="weight_loss_probability",
)

demo_recs

,recipe_id,name,final_description,nutri_score_label,macro_labels,macro_score,ingredient_score,weight_loss_score,weight_loss_probability,health_score,matched_ingredients,missing_ingredients,ingredients_list,instructions_list,food_tags,cook_time,calories,protein,fat,carbs
0,406080,Turkey Chili,This is my mom's chili recipe with a twist. I...,A,"[High Protein, Fiber Friendly, Low Sugar, Low ...",36.83,0.4077,3.8441,0.881975,72.260002,"[onion, tomato]","[chili powder, cumin, garlic powder, green bel...","[ground turkey breast, green bell pepper, onio...","[Place canned ingredients into Crockpot., Saut...","[chicken, poultry]",15.0,246.6,26.500000,2.9,33.3
1,214285,Low Fat Smoked Turkey and Lentil Salad,OMG! You HAVE to try this!\r\nThis recipe...pl...,A,"[High Protein, Fiber Friendly, Low Sugar, Low ...",14.14,0.4500,3.8399,0.881735,70.449997,"[garlic, olive oil, tomato]","[brown lentil, celery, coriander, cumin, grain...","[smoked turkey, brown lentils, garlic cloves, ...","[PLACE lentils in small saucepan, cover with w...","[chicken, dairy, poultry]",20.0,97.1,9.200000,1.8,11.7
2,355319,Hiller's Buffalo/Venison Chili,Hillers grocery store taught a cooking class a...,A,"[High Protein, Fiber Friendly, Low Sugar, Low ...",31.10,0.3636,3.8368,0.881557,72.290001,[tomato],"[black bean, black pepper, cayenne pepper, chi...","[ground fresh buffalo venison, can pinto beans...","[Brown meat with diced onion over low heat., A...","[vegan, vegetarian]",45.0,215.1,21.700001,2.6,27.6
3,368687,Creamy Black Bean Dip,I made this with regular cream cheese and a sh...,A,"[High Protein, Fiber Friendly, Low Sugar, Low ...",12.41,0.3875,3.8307,0.881206,70.279999,[tomato],"[baked corn tortilla chips, black bean, cayenn...","[fat cream cheese, black beans, reduced - fat ...","[In a food processor, cover and process cream ...","[dairy, vegetarian]",10.0,87.1,8.800000,0.8,11.4
4,290053,Easy Taco Soup,Very easy to make and doen't take very long a...,A,"[High Protein, Fiber Friendly, Low Sugar, Low ...",44.25,0.4167,3.8087,0.879932,73.209999,[tomato],"[pinto bean, pinto bean jalapeno pepper, ranch...","[pinto beans, pinto beans jalapeno peppers, ra...","[Brown Turkey Meat., In large pot mix all of t...","[chicken, poultry]",25.0,306.9,29.900000,3.9,40.5
